# SS316 暫用工程曲線：整理、插值與圖表
這份範例使用本機 CSV，不會下載或上傳研究資料。在 VS Code 右上角選擇專案 `.venv` 核心，再依序執行。

資料來自暫用圖像擷取，尚未驗證材料狀態與均勻變形範圍。只保留可辨識、應變不重複的連續資料段；不補初始段、不跨過遮蔽區域。這是探索性工程曲線，不是完整 SS316 塑性本構。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from feodosyev_shell.constitutive import curve_strain_intervals, evaluate_curve
from feodosyev_shell.materials.provisional import prepare_provisional_ss316
from feodosyev_shell.results import write_manifest
from feodosyev_shell.visualization import plot_curves_2d, plot_surface_3d

root = Path.cwd()
if not (root / "pyproject.toml").exists():
    root = root.parent
assert (root / "pyproject.toml").exists(), "請從專案或 notebooks 資料夾執行"
source = Path("D:/LAB/regen/references/SS316_data/SS316_plastic_StressStrain.csv")
material = prepare_provisional_ss316(source, root / "data/processed/ss316_provisional_v1")
output = root / "outputs/phase2_provisional"
output.mkdir(parents=True, exist_ok=True)
print(material.completeness_report())
print(material.metadata["selection_counts"])

## 1. 先看能查詢的範圍
應變 `0.2` 代表 20%，不是 0.2%。溫度輸入 K，應力輸出 Pa。
清單內每一對數字是一段可用範圍；段與段之間不能查詢。中間溫度只使用相鄰兩條曲線共同有資料的部分。

In [ ]:
temperature_k = 400 + 273.15
print("400 °C 可用應變範圍：", curve_strain_intervals(material, temperature_k))
stress = evaluate_curve(material, temperature_k, 0.2, kind="engineering", allow_provisional=True)
print("400 °C、20% 工程應變：", stress / 1e6, "MPa")

## 2. 核對原始節點與溫度中點
下方檢查數值插值是否正確，不能取代原始文獻的材料驗證。

In [ ]:
for temperature, (strain, stress) in material.curves.items():
    actual = evaluate_curve(
        material, temperature, strain, kind="engineering", allow_provisional=True
    )
    np.testing.assert_allclose(actual, stress, rtol=1e-8, atol=1e-6)
temperatures = sorted(material.curves)
rows = []
for low, high in zip(temperatures[:-1], temperatures[1:], strict=False):
    mid = (low + high) / 2
    for left, right in curve_strain_intervals(material, mid):
        strain = np.linspace(left, right, 5)
        expected = (
            np.interp(strain, *material.curves[low]) + np.interp(strain, *material.curves[high])
        ) / 2
        actual = evaluate_curve(material, mid, strain, kind="engineering", allow_provisional=True)
        np.testing.assert_allclose(actual, expected, rtol=1e-8, atol=1e-6)
        rows.extend(
            {
                "temperature_k": mid,
                "engineering_strain": float(x),
                "engineering_stress_pa": float(y),
            }
            for x, y in zip(strain, actual, strict=False)
        )
pd.DataFrame(rows).to_csv(output / "interpolated_samples.csv", index=False)
print("原始節點及相鄰溫度中點檢查通過")

## 3. 原始保留點與插值曲線
有圓點的是保留下來的原始節點，400 °C 的線是插值結果。未保留的原始點與原因完整記錄於 `selection_audit.csv`。

In [ ]:
figure_2d = plot_curves_2d(
    material, sorted([*material.curves, temperature_k]), allow_provisional=True
)
figure_2d.write_html(output / "curves_2d.html", include_plotlyjs=True)
figure_2d.show()

## 4. 三維圖與重現紀錄
拖曳旋轉、滾輪縮放、游標查值；白色缺口代表不可查詢，並非零應力。HTML 已包含繪圖程式，可離線開啟。

本資料不自動轉成真應力／真塑性應變。轉換公式的示範與測試請看 `01_material_curves.ipynb`，只有已確認均勻變形且彈性模數相容時才能套用。

In [ ]:
figure_3d = plot_surface_3d(material, allow_provisional=True)
figure_3d.write_html(output / "surface_3d.html", include_plotlyjs=True)
figure_3d.show()
manifest = write_manifest(
    material,
    output / "manifest.json",
    settings={
        "example_temperature_k": temperature_k,
        "example_strain": 0.2,
        "allow_provisional": True,
        "selection_policy": material.metadata["selection_policy"],
        "surface_n_temperature": 30,
        "surface_n_strain": 80,
    },
)
print("輸出位置：", output)
print("來源 SHA256：", manifest["source_sha256"])

## 後續使用
準備好的資料可以用 `load_material(root / 'data/processed/ss316_provisional_v1')` 重新讀取。
`ss316_local` 仍是先前的材料性質與熱膨脹資料；兩者沒有自動合併，因為來源與材料條件尚未確認相容。
取得作者正式數據後，建立新資料集並重新驗證，不覆蓋這份暫用資料。Phase 3 求解器尚未開始。